In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../'))

import numpy as np
import tensorflow as tf
import random

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

In [ ]:
import json
from src.pipeline.captioning_scratch import ImageCaptionerScratch
from src.evaluation.metrics import compute_bleu4, compute_meteor, timed_predict_from_features

VOCAB_PATH = '../../data/captions/vocab.json'
TEST_CAPTIONS_PATH = '../../data/captions/test_captions.json'
TEST_FEATURES_PATH = '../../data/features/test_features.npy'

with open(TEST_CAPTIONS_PATH) as f:
    test_data = json.load(f)

test_image_names = list(test_data.keys())
test_references  = [test_data[name] for name in test_image_names]

test_features = np.load(TEST_FEATURES_PATH, allow_pickle=True).item()
if isinstance(test_features, dict):
    test_feat_matrix = np.stack([test_features[n] for n in test_image_names])
else:
    test_feat_matrix = test_features

print(f'Test images: {len(test_image_names)}')
print(f'Feature matrix shape: {test_feat_matrix.shape}')

In [ ]:
VARIANTS = {
    # RNN
    'rnn_1layer_128':  {'type': 'rnn',  'weights': '../../weights/rnn_lstm/rnn_preinject_L1_H128.h5'},
    'rnn_2layer_128':  {'type': 'rnn',  'weights': '../../weights/rnn_lstm/rnn_preinject_L2_H128.h5'},
    'rnn_3layer_128':  {'type': 'rnn',  'weights': '../../weights/rnn_lstm/rnn_preinject_L3_H128.h5'},
    'rnn_1layer_512':  {'type': 'rnn',  'weights': '../../weights/rnn_lstm/rnn_preinject_L1_H512.h5'},
    'rnn_2layer_512':  {'type': 'rnn',  'weights': '../../weights/rnn_lstm/rnn_preinject_L2_H512.h5'},
    'rnn_3layer_512':  {'type': 'rnn',  'weights': '../../weights/rnn_lstm/rnn_preinject_L3_H512.h5'},
    # LSTM
    'lstm_1layer_128': {'type': 'lstm', 'weights': '../../weights/rnn_lstm/lstm_preinject_L1_H128.h5'},
    'lstm_2layer_128': {'type': 'lstm', 'weights': '../../weights/rnn_lstm/lstm_preinject_L2_H128.h5'},
    'lstm_3layer_128': {'type': 'lstm', 'weights': '../../weights/rnn_lstm/lstm_preinject_L3_H128.h5'},
    'lstm_1layer_512': {'type': 'lstm', 'weights': '../../weights/rnn_lstm/lstm_preinject_L1_H512.h5'},
    'lstm_2layer_512': {'type': 'lstm', 'weights': '../../weights/rnn_lstm/lstm_preinject_L2_H512.h5'},
    'lstm_3layer_512': {'type': 'lstm', 'weights': '../../weights/rnn_lstm/lstm_preinject_L3_H512.h5'},
}

In [ ]:
results = {}

for variant_name, config in VARIANTS.items():
    print(f'\n=== {variant_name} ===')
    captioner = ImageCaptionerScratch(decoder_type=config['type'])
    captioner.load_weights(
        cnn_encoder_path='InceptionV3',
        decoder_weights_path=config['weights'],
        vocab_path=VOCAB_PATH,
    )

    captions, total_time, avg_time = timed_predict_from_features(
        captioner, test_feat_matrix, max_len=20
    )
    bleu4  = compute_bleu4(test_references, captions)
    meteor = compute_meteor(test_references, captions)

    results[variant_name] = {
        'bleu4':             bleu4,
        'meteor':            meteor,
        'total_time':        total_time,
        'avg_time_per_image': avg_time,
        'captions':          captions,
    }
    print(f'  BLEU-4={bleu4:.4f}  METEOR={meteor:.4f}  avg={avg_time:.3f}s')

In [ ]:
import json, pandas as pd
import os

os.makedirs('../../results/tables', exist_ok=True)

# Save scores (without raw captions)
scores_only = {
    k: {kk: vv for kk, vv in v.items() if kk != 'captions'}
    for k, v in results.items()
}
with open('../../results/tables/all_variants_scores.json', 'w') as f:
    json.dump(scores_only, f, indent=2)

# Summary table
df = pd.DataFrame([
    {
        'variant':    k,
        'type':       'RNN' if 'rnn' in k else 'LSTM',
        'bleu4':      round(v['bleu4'], 4),
        'meteor':     round(v['meteor'], 4),
        'avg_time_s': round(v['avg_time_per_image'], 3),
    }
    for k, v in results.items()
]).sort_values('bleu4', ascending=False)

display(df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid', palette='muted')
os.makedirs('../../results/plots', exist_ok=True)

names  = list(results.keys())
bleu4s = [results[n]['bleu4'] for n in names]
times  = [results[n]['avg_time_per_image'] for n in names]
colors = ['#4C72B0' if 'rnn' in n else '#DD8452' for n in names]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# BLEU-4
bars = axes[0].bar(names, bleu4s, color=colors)
axes[0].set_title('BLEU-4 Score — All Variants', fontsize=14)
axes[0].set_ylabel('BLEU-4', fontsize=12)
axes[0].set_xticklabels(names, rotation=45, ha='right')
for bar, val in zip(bars, bleu4s):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=8)

# Avg time
bars2 = axes[1].bar(names, times, color=colors)
axes[1].set_title('Avg Inference Time per Image — All Variants', fontsize=14)
axes[1].set_ylabel('Time (s)', fontsize=12)
axes[1].set_xticklabels(names, rotation=45, ha='right')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#4C72B0', label='RNN'),
                   Patch(facecolor='#DD8452', label='LSTM')]
axes[0].legend(handles=legend_elements)
axes[1].legend(handles=legend_elements)

plt.tight_layout()
plt.savefig('../../results/plots/bleu4_and_time_all_variants.png', dpi=150, bbox_inches='tight')
plt.show()